# NPS Experiment 012 — Policy Subspace Causal Necessity (Probe-Ablation Test)

**Neural Privilege Separation (NPS) research series**

**Research question.** Experiment 011 found that `Qwen/Qwen2.5-1.5B-Instruct`
encodes benign-vs-unsafe policy information in a low-dimensional subspace
(intrinsic dimensionality ~14-35 components at 90% variance depending on
layer, U-shaped across depth), and that — once within-class topic variance
is controlled for via between-class (paired-difference) PCA — the
class-separating direction increasingly aligns with PC1 at depth. This
notebook asks the natural follow-up:

> **Is that subspace causally necessary for the *linear separability* of
> policy-relevant activations — or is the same information redundantly
> encoded elsewhere in the residual stream, such that removing the
> subspace doesn't actually destroy it?**

**Scope, deliberately narrow.** This is a **representational** causal test,
not a behavioral one. We do not generate text, evaluate jailbreak success,
or measure refusal rate — this notebook only asks whether *linear probes*
can still separate benign/unsafe activations after the discovered subspace
is projected out. That's a meaningfully different (and narrower) claim than
"the model can be made to comply with unsafe requests," and the two should
not be conflated when interpreting results below.

**Method, in one paragraph.** For each representative layer (early, middle,
final) and each subspace definition from Experiment 011 (within-class PCA,
between-class/paired-difference PCA), we project the top-*k* components out
of held-out benign/unsafe activations for *k* ∈ {1, 2, 5, 10, 20}, then
measure two things: (a) whether the **original, frozen** Experiment-011
probe still classifies the ablated activations correctly ("did we destroy
the direction the probe was actually using?"), and (b) whether a **freshly
re-fit** probe trained on the ablated activations can still separate the
classes ("is the information gone, or just moved / redundantly encoded?").
Both are compared against ablating a random *k*-dimensional subspace of the
same size, so we can tell subspace-specific effects from generic
dimensionality-reduction effects.

---


## 0. Runtime check

No text generation happens in this notebook, so a GPU is optional — it is
only used as a fallback if some Experiment 011 activation cache is missing
and needs to be (re-)extracted. If you're confident all caches are present,
you can run this notebook on CPU-only Colab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv 2>/dev/null || echo "No GPU detected (fine unless activation re-extraction is needed)."
import torch
print("CUDA available:", torch.cuda.is_available())


## 1. Setup

Installs pinned dependencies (matplotlib pinned to a specific stable
release, matching the convention adopted in the Experiment 011 update) and
mounts Google Drive. We resolve **two** base directories: this experiment's
own `BASE_DIR` (for its own resumable outputs) and `PRIOR_BASE_DIR`,
pointing at Experiment 011's results — that's where cached activations,
probes, and PCA subspaces are reused from.

In [ ]:
%%capture
# matplotlib pinned to 3.8.4 / seaborn to 0.13.2 — same stable pin used in the
# Experiment 011 notebook update, avoiding intermittent Colab rendering issues
# seen on newer matplotlib releases.
!pip install -q -U "transformers>=4.46" "accelerate>=0.34" "scikit-learn>=1.4" \
    "matplotlib==3.8.4" "seaborn==0.13.2" "pandas>=2.2" "scipy>=1.11" "tqdm" "huggingface_hub"


In [ ]:
import os

USE_DRIVE = True  # set False to keep everything local to the Colab VM

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_NPS_ROOT = '/content/drive/MyDrive/NPS'
    except Exception as e:
        print("Drive mount failed or not in Colab, falling back to local storage:", e)
        DRIVE_NPS_ROOT = None
else:
    DRIVE_NPS_ROOT = None

# --- this experiment's own resumable output directory ---
BASE_DIR = (os.path.join(DRIVE_NPS_ROOT, 'exp012_policy_subspace_intervention')
            if DRIVE_NPS_ROOT else '/content/nps_exp012_policy_subspace_intervention')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')
ABLATION_DIR = os.path.join(RESULTS_DIR, 'ablation')
STATS_DIR    = os.path.join(RESULTS_DIR, 'statistics')
FIGURES_DIR  = os.path.join(RESULTS_DIR, 'figures')
for d in [RESULTS_DIR, ABLATION_DIR, STATS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Experiment 011's output directory, where we reuse activations/probes/PCA ---
PRIOR_RESULTS_DIR_OVERRIDE = None  # set explicitly if your Exp011 results live somewhere else

def resolve_prior_results_dir():
    if PRIOR_RESULTS_DIR_OVERRIDE:
        return PRIOR_RESULTS_DIR_OVERRIDE
    candidates = []
    if DRIVE_NPS_ROOT:
        candidates.append(os.path.join(DRIVE_NPS_ROOT, 'exp011_policy_subspace', 'results'))
    candidates.append('/content/nps_exp011_policy_subspace/results')
    for c in candidates:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, 'activations')):
            return c
    raise FileNotFoundError(
        "Could not locate Experiment 011 results (looked in: "
        f"{candidates}). Set PRIOR_RESULTS_DIR_OVERRIDE to the exp011 `results/` "
        "folder (e.g. a Drive path) before continuing."
    )

PRIOR_RESULTS_DIR = resolve_prior_results_dir()
PRIOR_ACTS_DIR     = os.path.join(PRIOR_RESULTS_DIR, 'activations')
PRIOR_PROBES_DIR   = os.path.join(PRIOR_RESULTS_DIR, 'probes')
PRIOR_SUBSPACE_DIR = os.path.join(PRIOR_RESULTS_DIR, 'subspace')

print("This experiment's outputs:", RESULTS_DIR)
print("Reusing Experiment 011 artifacts from:", PRIOR_RESULTS_DIR)


## 2. Configuration

`seed`, `test_size`, `probe_C`, and `probe_max_iter` are set to match
Experiment 011 exactly — this lets us reproduce the *same* train/test split
(via `train_test_split` with the same `random_state`) so the "frozen probe"
evaluation is on the identical held-out set the original probe was scored
on.

In [ ]:
from dataclasses import dataclass, field
from typing import List
import json
import torch as _torch_probe

@dataclass
class Config:
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"  # only used as extraction fallback
    dtype: str = "float16"
    device: str = "cuda" if _torch_probe.cuda.is_available() else "cpu"
    seed: int = 42                 # must match Exp011's config.seed
    test_size: float = 0.2         # must match Exp011's config.test_size
    probe_C: float = 1.0           # must match Exp011's config.probe_C (used ONLY to reproduce the original probe's split/metrics)
    probe_max_iter: int = 2000     # must match Exp011's config.probe_max_iter
    refit_probe_C: float = 0.01    # MUCH stronger L2 than probe_C: with n_train~320 and
                                    # hidden_size~1536, ablation leaves us deep in a p>>n
                                    # regime where any labeling is perfectly linearly
                                    # separable by chance; weak regularization (C=1.0) fits
                                    # that chance structure and can generalize far worse than
                                    # random. C=0.01 keeps the re-fit probe honest.

    k_list: List[int] = field(default_factory=lambda: [1, 2, 5, 10, 20])
    subspace_types: List[str] = field(default_factory=lambda: ["within", "between"])
    n_random_trials: int = 20      # random-subspace control trials per (layer, k)
    n_bootstrap: int = 2000        # bootstrap resamples for accuracy CIs
    ci_level: float = 0.95

config = Config()
print(json.dumps(config.__dict__, indent=2, default=str))


## 3. Stage & checkpoint utilities

Identical conventions to Experiment 011: a JSON stage manifest gates each
major stage so re-running the notebook after a disconnect skips completed
work; every stage saves its output immediately and then cleans up.

In [ ]:
import gc, time, pickle, json, os
import numpy as np
import torch

STAGE_FILE = os.path.join(RESULTS_DIR, "stage_manifest.json")

def _load_manifest():
    if os.path.exists(STAGE_FILE):
        with open(STAGE_FILE) as f:
            return json.load(f)
    return {}

def _save_manifest(m):
    with open(STAGE_FILE, "w") as f:
        json.dump(m, f, indent=2)

def stage_done(name):
    return _load_manifest().get(name, {}).get("done", False)

def mark_done(name, meta=None):
    m = _load_manifest()
    m[name] = {"done": True, "timestamp": time.time(), "meta": meta or {}}
    _save_manifest(m)
    print(f"[stage] '{name}' marked complete.")

def cleanup(*tensors):
    for t in tensors:
        try:
            del t
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

print("Stage manifest at:", STAGE_FILE)
print("Existing stages:", list(_load_manifest().keys()))


## 4. Load Experiment 011 artifacts

Reuses cached activations, trained probes, and both PCA subspace variants
(within-class from Section 8, between-class/paired-difference from Section
8b) from Experiment 011. If an activation cache is missing for a layer we
need, we fall back to re-extracting it with the same hook-based method —
this is the *only* place this notebook touches the model or the GPU.

In [ ]:
def act_path(layer, cls):
    return os.path.join(PRIOR_ACTS_DIR, f"layer{layer:02d}_{cls}.npy")

def probe_path(layer):
    return os.path.join(PRIOR_PROBES_DIR, f"probe_layer{layer:02d}.pkl")

def within_pca_path(layer):
    return os.path.join(PRIOR_SUBSPACE_DIR, f"pca_layer{layer:02d}.pkl")

def between_pca_path(layer):
    return os.path.join(PRIOR_SUBSPACE_DIR, f"pca_betweenclass_layer{layer:02d}.pkl")

with open(os.path.join(PRIOR_RESULTS_DIR, "stage_manifest.json")) as f:
    prior_manifest = json.load(f)
prior_layers = sorted(prior_manifest.get("activation_extraction", {}).get("meta", {}).get("layers", []))
print("Layers with cached activations in Experiment 011:", prior_layers)

with open(os.path.join(PRIOR_PROBES_DIR, "probe_metrics.json")) as f:
    prior_probe_metrics = json.load(f)

# Representative early/middle/final layers MUST match the specific layers
# Experiment 011 designated as "early"/"middle"/"final" (via early_frac /
# middle_frac / final_frac) — these are NOT simply the min/max of the
# analyzed layer sweep, which instead spans the full `layer_fracs` grid
# (typically including layer 0 and the last layer, neither of which need be
# the same as the early/final *representative* layers). Reading Exp011's own
# `early_middle_final_comparison.csv` is the reliable source of truth; only
# fall back to recomputing from fracs if that file is missing.
emf_path = os.path.join(PRIOR_SUBSPACE_DIR, "early_middle_final_comparison.csv")
if os.path.exists(emf_path):
    import pandas as _pd
    _emf = _pd.read_csv(emf_path)
    rep_layers = {row.regime: int(row.layer) for row in _emf.itertuples()}
else:
    num_layers = max(prior_layers) + 1
    rep_layers = {
        "early":  int(round(0.15 * (num_layers - 1))),
        "middle": int(round(0.50 * (num_layers - 1))),
        "final":  int(round(0.92 * (num_layers - 1))),
    }
    print("Warning: early_middle_final_comparison.csv not found; "
          "recomputed rep_layers from default fracs — verify these match "
          "Experiment 011's actual config if it used non-default fracs.")
print("Representative layers for ablation study:", rep_layers)


In [ ]:
_model = None
_tokenizer = None

def get_model_and_tokenizer():
    '''Lazy-load fallback, only used if an activation cache is unexpectedly
    missing for one of the representative layers.'''
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"[fallback] Loading {config.model_name} to re-extract missing activations...")
    _tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    if _tokenizer.pad_token is None:
        _tokenizer.pad_token = _tokenizer.eos_token
    _model = AutoModelForCausalLM.from_pretrained(
        config.model_name, torch_dtype=torch.float16, device_map=config.device, low_cpu_mem_usage=True)
    _model.eval()
    return _model, _tokenizer

def extract_activations_fallback(prompts, cls, layer, batch_size=8):
    '''Minimal re-implementation of Exp011's extraction, used only if a
    required cache file is missing.'''
    model, tokenizer = get_model_and_tokenizer()
    captured = {}
    def hook(module, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        captured["h"] = hs.detach()
    h = model.model.layers[layer].register_forward_hook(hook)
    buf = []
    try:
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i + batch_size]
            msgs = [[{"role": "user", "content": p}] for p in batch]
            texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(config.device)
            with torch.no_grad():
                model(**enc)
            attn = enc["attention_mask"]
            last_idx = attn.sum(dim=1) - 1
            pooled = captured["h"][torch.arange(captured["h"].size(0)), last_idx].float().cpu().numpy()
            buf.append(pooled)
            cleanup(enc)
    finally:
        h.remove()
    arr = np.concatenate(buf, axis=0)
    np.save(act_path(layer, cls), arr)
    return arr

def load_layer_activations(layer):
    b_path, u_path = act_path(layer, "benign"), act_path(layer, "unsafe")
    if not (os.path.exists(b_path) and os.path.exists(u_path)):
        raise FileNotFoundError(
            f"Missing cached activations for layer {layer} and no prompt set available "
            f"to re-extract in this notebook. Point PRIOR_RESULTS_DIR_OVERRIDE at a "
            f"results folder that has them, or re-run Experiment 011 for this layer."
        )
    return np.load(b_path), np.load(u_path)

def backfill_betweenclass_pca(layer, n_shuffles=5, seed=42):
    '''Some Experiment 011 runs only produced within-class PCA and skipped
    the between-class (paired-difference) variant. Rather than hard-failing,
    reconstruct it here from the cached activations/probe that ARE present —
    same method as Experiment 011 Section 8b (uncentered SVD on paired
    unsafe-benign difference vectors, several random pairings).'''
    out_path = between_pca_path(layer)
    if os.path.exists(out_path):
        return
    benign, unsafe = load_layer_activations(layer)
    n = min(len(benign), len(unsafe))
    rng = np.random.default_rng(seed + layer)
    diffs = [unsafe[:n] - benign[rng.permutation(n)][:n] for _ in range(n_shuffles)]
    D = np.concatenate(diffs, axis=0)
    max_rank = min(128, D.shape[0], D.shape[1])
    U, S, Vt = np.linalg.svd(D, full_matrices=False)
    Vt, S = Vt[:max_rank], S[:max_rank]
    evr = (S ** 2) / np.sum(S ** 2)
    cum = np.cumsum(evr)
    probe = load_pickle(probe_path(layer))

    def _cos(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

    result = {
        "layer": layer, "explained_variance_ratio": evr, "cumulative_explained_variance": cum,
        "variance_at_k": {k: float(cum[k - 1]) if k <= len(cum) else float(cum[-1]) for k in [1, 2, 5, 10, 20, 50]},
        "intrinsic_dim": {t: int(min(int(np.searchsorted(cum, t)) + 1, max_rank)) for t in [0.8, 0.9, 0.95, 0.99]},
        "components": Vt[:50], "pc1_vs_probe_cosine": _cos(Vt[0], probe["weight"]),
        "pc1_vs_meandiff_cosine": _cos(Vt[0], D.mean(axis=0)),
        "max_rank": max_rank, "n_diff_vectors": D.shape[0], "hidden_size": D.shape[1],
    }
    save_pickle(result, out_path)
    cleanup(benign, unsafe, diffs, D, U, S, Vt, probe)
    print(f"[backfill] Reconstructed missing between-class PCA cache for layer {layer}.")

# Sanity check all representative layers are available before doing any work.
# Activations, the frozen probe, and the within-class PCA are hard requirements.
# The between-class PCA is auto-backfilled if missing and "between" is in use.
for name, l in rep_layers.items():
    if not (os.path.exists(act_path(l, "benign")) and os.path.exists(act_path(l, "unsafe"))):
        raise FileNotFoundError(f"Representative layer '{name}' (L{l}) has no cached activations at {PRIOR_ACTS_DIR}.")
    if not os.path.exists(probe_path(l)):
        raise FileNotFoundError(f"Representative layer '{name}' (L{l}) has no cached probe at {PRIOR_PROBES_DIR}.")
    if not os.path.exists(within_pca_path(l)):
        raise FileNotFoundError(f"Representative layer '{name}' (L{l}) is missing the within-class PCA cache at {PRIOR_SUBSPACE_DIR}.")
    if "between" in config.subspace_types and not os.path.exists(between_pca_path(l)):
        backfill_betweenclass_pca(l)
print("All required Experiment 011 artifacts found for representative layers:", rep_layers)


## 5. Ablation and evaluation machinery

- `project_out(H, V)` removes the span of the (orthonormal) rows of `V`
  from each row of `H`: `H' = H - (H·Vᵀ)·V`.
- `eval_frozen_probe` applies the **original Experiment-011 probe's fixed
  weight vector AND intercept** to (possibly ablated) activations. **Caveat
  worth understanding before trusting this number:** if the removed
  component carries much of the data's overall mean, ablation rigidly
  shifts every score by a constant — the fixed intercept, calibrated for
  the *original* distribution, can become miscalibrated and cause a
  degenerate constant-class prediction (accuracy pinned at exactly the
  test set's class balance) even when the underlying direction still ranks
  the classes correctly. Always read this alongside `eval_frozen_recalibrated`
  and the AUC column, not accuracy in isolation.
- `eval_frozen_recalibrated` keeps the probe's **direction** completely
  frozen but refits just a 1-D scale + intercept on the ablated training
  projections — this isolates "does the original direction still separate
  the classes" from "is the original threshold still appropriate," which
  `eval_frozen_probe` alone conflates.
- `eval_refit_probe` trains a **brand-new** logistic regression on the
  (possibly ablated) activations, with `config.refit_probe_C` (strong L2)
  rather than the original probe's `C`. This matters: with `n_train~320`
  and `hidden_size~1536`, ablation leaves a `p >> n` regime where *any*
  labeling of the training data is perfectly linearly separable by chance,
  so weak regularization fits training noise and can generalize **worse
  than chance** on held-out data — which would otherwise look like (false)
  evidence the information was destroyed, rather than an overfitting
  artifact.
- `bootstrap_accuracy_ci` and `mcnemar_test` give the statistics required to
  say whether an observed accuracy drop is meaningful rather than noise.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.stats import chi2

def project_out(H, V):
    if V is None or V.shape[0] == 0:
        return H
    coords = H @ V.T
    return H - coords @ V

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def get_split(layer):
    '''Reproduces Experiment 011's exact train/test split (same seed,
    same test_size, same X/y construction order) so frozen-probe evaluation
    is on the identical held-out set the original probe was scored on.'''
    benign, unsafe = load_layer_activations(layer)
    X = np.concatenate([benign, unsafe], axis=0)
    y = np.concatenate([np.zeros(len(benign)), np.ones(len(unsafe))])
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=config.test_size, random_state=config.seed, stratify=y)
    return Xtr, Xte, ytr, yte

def eval_frozen_probe(weight, bias, X, y):
    '''Raw evaluation using the ORIGINAL probe's weight AND intercept,
    unchanged. Caveat (see markdown above): projecting out a component that
    carries much of the data's mean rigidly shifts every score by a
    constant, so this fixed intercept can become miscalibrated after
    ablation and cause a degenerate constant-class prediction even when the
    underlying direction still ranks the classes correctly. Always sanity
    check this against eval_frozen_recalibrated and the AUC column before
    concluding "no separability survives".'''
    z = X @ weight + bias
    proba = sigmoid(z)
    pred = (proba >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    auc = roc_auc_score(y, proba) if len(set(y)) > 1 else float("nan")
    return acc, auc, pred, proba

def eval_frozen_recalibrated(weight, Xtr, ytr, Xte, yte):
    '''Keeps the probe's DIRECTION completely frozen (never refit), but
    recalibrates the 1-D scale + intercept on the (possibly ablated)
    training projections. This isolates "does the original direction still
    separate the classes" from "is the original fixed threshold still
    appropriate for this distribution" — the two get conflated by
    eval_frozen_probe once ablation shifts the score distribution. Fitting
    a 1-D logistic regression on ~300 scalar values is not a meaningful
    overfitting risk (unlike re-fitting in the full ~1500-D space).'''
    str_ = (Xtr @ weight).reshape(-1, 1)
    ste_ = (Xte @ weight).reshape(-1, 1)
    recal = LogisticRegression(max_iter=1000)
    recal.fit(str_, ytr)
    pred = recal.predict(ste_)
    proba = recal.predict_proba(ste_)[:, 1]
    acc = accuracy_score(yte, pred)
    auc = roc_auc_score(yte, proba) if len(set(yte)) > 1 else float("nan")
    cleanup(recal)
    return acc, auc, pred, proba

def eval_refit_probe(Xtr, ytr, Xte, yte):
    '''Trains a fresh classifier on the (possibly ablated) activations.
    Uses config.refit_probe_C (strong L2), NOT config.probe_C — see the
    Config docstring: with n_train~320 and hidden_size~1536, weak
    regularization in this p>>n regime fits training noise and can
    generalize far worse than chance, which would be mistaken for evidence
    the information was destroyed rather than an artifact of overfitting.'''
    clf = LogisticRegression(C=config.refit_probe_C, max_iter=config.probe_max_iter)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:, 1]
    acc = accuracy_score(yte, pred)
    auc = roc_auc_score(yte, proba)
    cleanup(clf)
    return acc, auc, pred, proba

def bootstrap_accuracy_ci(correct_binary, n_boot=None, seed=0):
    n_boot = n_boot or config.n_bootstrap
    rng = np.random.default_rng(seed)
    n = len(correct_binary)
    means = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        means[i] = correct_binary[idx].mean()
    lo, hi = np.percentile(means, [(1 - config.ci_level) / 2 * 100, (1 + config.ci_level) / 2 * 100])
    return float(correct_binary.mean()), float(lo), float(hi)

def mcnemar_test(baseline_correct, ablated_correct):
    '''Paired test on the same held-out instances: did ablation change which
    instances the probe got right, more than chance?'''
    b = int(np.sum(baseline_correct & ~ablated_correct))   # correct -> incorrect
    c = int(np.sum(~baseline_correct & ablated_correct))   # incorrect -> correct
    if b + c == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / (b + c)   # continuity-corrected McNemar
        p = float(1 - chi2.cdf(stat, 1))
    return {"b_correct_to_incorrect": b, "c_incorrect_to_correct": c,
            "statistic": float(stat), "p_value": p}

def cohens_h(p1, p2):
    '''Effect size for the difference between two proportions (accuracies).'''
    return float(2 * np.arcsin(np.sqrt(np.clip(p1, 0, 1))) - 2 * np.arcsin(np.sqrt(np.clip(p2, 0, 1))))

def random_orthonormal_subspace(hidden_size, k, rng):
    A = rng.normal(size=(hidden_size, k))
    Q, _ = np.linalg.qr(A)
    return Q.T  # [k, hidden_size], rows orthonormal


## 6. Ablation sweep

For every `k` in `config.k_list`, and for every representative layer and
subspace type, we ablate the top-*k* components, evaluate frozen + re-fit
probes, run the random-subspace control, and **save `results_k{k}.csv`
immediately** — matching the incremental-save convention from Experiment
011. Nothing generation-related happens here; the arrays involved are all
small (≤ a few hundred rows × 1536 dims), so memory pressure is minimal,
but we still `del` + `gc.collect()` per k as a matter of house style.

In [ ]:
ABLATION_STAGE = "policy_subspace_ablation_sweep"

def run_ablation_for_k(k):
    rows = []
    for layer_name, layer in rep_layers.items():
        Xtr, Xte, ytr, yte = get_split(layer)
        probe = load_pickle(probe_path(layer))
        w, b = probe["weight"], probe["bias"]

        # unablated baselines (k=0), needed for McNemar pairing and effect sizes.
        # Both the raw-threshold and recalibrated baselines should agree closely
        # since nothing has been ablated yet — a good sanity check in itself.
        base_acc, base_auc, base_pred, base_proba = eval_frozen_probe(w, b, Xte, yte)
        base_correct = (base_pred == yte)
        base_recal_acc, base_recal_auc, base_recal_pred, _ = eval_frozen_recalibrated(w, Xtr, ytr, Xte, yte)
        base_recal_correct = (base_recal_pred == yte)

        for subspace_type in config.subspace_types:
            pca = load_pickle(within_pca_path(layer) if subspace_type == "within" else between_pca_path(layer))
            V = pca["components"][:k]

            Xtr_abl = project_out(Xtr, V)
            Xte_abl = project_out(Xte, V)

            f_acc, f_auc, f_pred, f_proba = eval_frozen_probe(w, b, Xte_abl, yte)
            f_correct = (f_pred == yte)
            f_mean, f_lo, f_hi = bootstrap_accuracy_ci(f_correct.astype(int), seed=config.seed + k)
            f_mcnemar = mcnemar_test(base_correct, f_correct)
            f_effect = cohens_h(f_acc, base_acc)

            fr_acc, fr_auc, fr_pred, fr_proba = eval_frozen_recalibrated(w, Xtr_abl, ytr, Xte_abl, yte)
            fr_correct = (fr_pred == yte)
            fr_mean, fr_lo, fr_hi = bootstrap_accuracy_ci(fr_correct.astype(int), seed=config.seed + k + 2)
            fr_mcnemar = mcnemar_test(base_recal_correct, fr_correct)
            fr_effect = cohens_h(fr_acc, base_recal_acc)

            r_acc, r_auc, r_pred, r_proba = eval_refit_probe(Xtr_abl, ytr, Xte_abl, yte)
            r_correct = (r_pred == yte)
            r_mean, r_lo, r_hi = bootstrap_accuracy_ci(r_correct.astype(int), seed=config.seed + k + 1)
            r_mcnemar = mcnemar_test(base_correct, r_correct)
            r_effect = cohens_h(r_acc, base_acc)

            rows.append({
                "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": subspace_type,
                "eval_type": "frozen_probe", "accuracy": f_acc, "auc": f_auc,
                "ci_lo": f_lo, "ci_hi": f_hi, "baseline_accuracy": base_acc,
                "mcnemar_p": f_mcnemar["p_value"], "cohens_h": f_effect,
                "n_test": len(yte),
            })
            rows.append({
                "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": subspace_type,
                "eval_type": "frozen_recalibrated", "accuracy": fr_acc, "auc": fr_auc,
                "ci_lo": fr_lo, "ci_hi": fr_hi, "baseline_accuracy": base_recal_acc,
                "mcnemar_p": fr_mcnemar["p_value"], "cohens_h": fr_effect,
                "n_test": len(yte),
            })
            rows.append({
                "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": subspace_type,
                "eval_type": "refit_probe", "accuracy": r_acc, "auc": r_auc,
                "ci_lo": r_lo, "ci_hi": r_hi, "baseline_accuracy": base_acc,
                "mcnemar_p": r_mcnemar["p_value"], "cohens_h": r_effect,
                "n_test": len(yte),
            })
            cleanup(Xtr_abl, Xte_abl)

        # random-subspace control (k-dim, same layer, n_random_trials draws)
        rng = np.random.default_rng(config.seed + 1000 * k + layer)
        f_accs, fr_accs, r_accs = [], [], []
        hidden_size = Xtr.shape[1]
        for t in range(config.n_random_trials):
            V_rand = random_orthonormal_subspace(hidden_size, k, rng)
            Xte_r = project_out(Xte, V_rand)
            Xtr_r = project_out(Xtr, V_rand)
            f_acc_r, _, _, _ = eval_frozen_probe(w, b, Xte_r, yte)
            fr_acc_r, _, _, _ = eval_frozen_recalibrated(w, Xtr_r, ytr, Xte_r, yte)
            r_acc_r, _, _, _ = eval_refit_probe(Xtr_r, ytr, Xte_r, yte)
            f_accs.append(f_acc_r)
            fr_accs.append(fr_acc_r)
            r_accs.append(r_acc_r)
            cleanup(Xte_r, Xtr_r)
        rows.append({
            "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": "random_control",
            "eval_type": "frozen_probe", "accuracy": float(np.mean(f_accs)), "auc": float("nan"),
            "ci_lo": float(np.percentile(f_accs, 2.5)), "ci_hi": float(np.percentile(f_accs, 97.5)),
            "baseline_accuracy": base_acc, "mcnemar_p": float("nan"),
            "cohens_h": cohens_h(float(np.mean(f_accs)), base_acc), "n_test": len(yte),
        })
        rows.append({
            "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": "random_control",
            "eval_type": "frozen_recalibrated", "accuracy": float(np.mean(fr_accs)), "auc": float("nan"),
            "ci_lo": float(np.percentile(fr_accs, 2.5)), "ci_hi": float(np.percentile(fr_accs, 97.5)),
            "baseline_accuracy": base_recal_acc, "mcnemar_p": float("nan"),
            "cohens_h": cohens_h(float(np.mean(fr_accs)), base_recal_acc), "n_test": len(yte),
        })
        rows.append({
            "k": k, "layer_name": layer_name, "layer": layer, "subspace_type": "random_control",
            "eval_type": "refit_probe", "accuracy": float(np.mean(r_accs)), "auc": float("nan"),
            "ci_lo": float(np.percentile(r_accs, 2.5)), "ci_hi": float(np.percentile(r_accs, 97.5)),
            "baseline_accuracy": base_acc, "mcnemar_p": float("nan"),
            "cohens_h": cohens_h(float(np.mean(r_accs)), base_acc), "n_test": len(yte),
        })

        cleanup(Xtr, Xte, ytr, yte, probe)

    return rows

import pandas as pd
from tqdm.auto import tqdm

if stage_done(ABLATION_STAGE):
    print("Ablation sweep already complete — loading cached per-k CSVs.")
    all_rows_df = pd.concat([pd.read_csv(os.path.join(ABLATION_DIR, f"results_k{k}.csv")) for k in config.k_list],
                             ignore_index=True)
else:
    per_k_dfs = []
    for k in tqdm(config.k_list, desc="ablation sweep over k"):
        out_path = os.path.join(ABLATION_DIR, f"results_k{k}.csv")
        if os.path.exists(out_path):
            df_k = pd.read_csv(out_path)
        else:
            rows = run_ablation_for_k(k)
            df_k = pd.DataFrame(rows)
            df_k.to_csv(out_path, index=False)  # save immediately, per k, as required
            print(f"Saved {out_path} ({len(df_k)} rows)")
        per_k_dfs.append(df_k)
        gc.collect()
    all_rows_df = pd.concat(per_k_dfs, ignore_index=True)
    mark_done(ABLATION_STAGE, meta={"k_list": config.k_list, "layers": rep_layers})

all_rows_df.to_csv(os.path.join(ABLATION_DIR, "results_all_k.csv"), index=False)
all_rows_df.head(12)


## 7. Aggregate statistics

Writes `statistics.json` summarizing, per layer and subspace type, the
accuracy drop from baseline, bootstrap CIs, McNemar significance, and
effect size, plus a comparison against the random-subspace control (a
subspace-specific effect should exceed what removing a random *k*-dim
subspace does).

In [ ]:
STATS_STAGE = "policy_subspace_ablation_statistics"

def build_statistics(df):
    stats = {}
    for layer_name, layer in rep_layers.items():
        stats[layer_name] = {"layer": layer, "by_k": {}}
        for k in config.k_list:
            sub = df[(df.layer_name == layer_name) & (df.k == k)]
            entry = {}
            for subspace_type in config.subspace_types + ["random_control"]:
                s2 = sub[sub.subspace_type == subspace_type]
                entry[subspace_type] = {
                    row.eval_type: {
                        "accuracy": row.accuracy, "auc": row.auc,
                        "ci_lo": row.ci_lo, "ci_hi": row.ci_hi,
                        "baseline_accuracy": row.baseline_accuracy,
                        "accuracy_drop": row.baseline_accuracy - row.accuracy,
                        "mcnemar_p": row.mcnemar_p, "cohens_h": row.cohens_h,
                    } for row in s2.itertuples()
                }
            stats[layer_name]["by_k"][str(k)] = entry
    return stats

if stage_done(STATS_STAGE):
    with open(os.path.join(STATS_DIR, "statistics.json")) as f:
        statistics = json.load(f)
    print("Statistics already computed — loaded from cache.")
else:
    statistics = build_statistics(all_rows_df)
    with open(os.path.join(STATS_DIR, "statistics.json"), "w") as f:
        json.dump(statistics, f, indent=2, default=float)
    mark_done(STATS_STAGE)

print(json.dumps(statistics["middle"]["by_k"].get(str(config.k_list[len(config.k_list)//2]), {}), indent=2)[:1500])


## 8. Publication-quality figures

Saved as `.png` (300 dpi) and `.pdf` under `results/figures/`, matching the
Experiment 011 convention.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{name}.pdf"), bbox_inches="tight")
    plt.close(fig)
    print("Saved figure:", name)

layer_order = list(rep_layers.keys())
layer_colors = dict(zip(layer_order, sns.color_palette("viridis", n_colors=len(layer_order))))


In [ ]:
# Fig 1: frozen-probe accuracy vs k removed, one panel per subspace type,
# one line per layer, random-control band shown for comparison.
fig, axes = plt.subplots(1, len(config.subspace_types), figsize=(6 * len(config.subspace_types), 4.5), sharey=True)
if len(config.subspace_types) == 1:
    axes = [axes]
for ax, subspace_type in zip(axes, config.subspace_types):
    for layer_name in layer_order:
        sub = all_rows_df[(all_rows_df.layer_name == layer_name) &
                           (all_rows_df.subspace_type == subspace_type) &
                           (all_rows_df.eval_type == "frozen_probe")].sort_values("k")
        ax.plot(sub.k, sub.accuracy, marker="o", label=f"{layer_name} (L{rep_layers[layer_name]})",
                color=layer_colors[layer_name])
        ax.fill_between(sub.k, sub.ci_lo, sub.ci_hi, alpha=0.15, color=layer_colors[layer_name])
        rand = all_rows_df[(all_rows_df.layer_name == layer_name) &
                            (all_rows_df.subspace_type == "random_control") &
                            (all_rows_df.eval_type == "frozen_probe")].sort_values("k")
        ax.plot(rand.k, rand.accuracy, linestyle="--", color=layer_colors[layer_name], alpha=0.5)
    ax.set_title(f"{subspace_type}-class subspace")
    ax.set_xlabel("Principal components removed (k)")
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Frozen-probe accuracy\n(original Exp. 011 weight vector)")
axes[-1].legend(fontsize=8, loc="lower left")
fig.suptitle("Does removing the policy subspace break the ORIGINAL probe's decision boundary?\n(dashed = random k-dim subspace control)")
save_fig(fig, "fig1_frozen_probe_accuracy_vs_k")


In [ ]:
# Fig 1b: RECALIBRATED frozen-direction accuracy vs k — companion to Fig 1.
# Same frozen weight vector, but threshold/scale recalibrated per-k, so this
# isolates whether the ORIGINAL DIRECTION still separates the classes,
# independent of whether ablation shifted the score distribution enough to
# break the original fixed threshold (which Fig 1 alone cannot distinguish
# from genuine loss of separability — see Section 5 markdown).
fig, axes = plt.subplots(1, len(config.subspace_types), figsize=(6 * len(config.subspace_types), 4.5), sharey=True)
if len(config.subspace_types) == 1:
    axes = [axes]
for ax, subspace_type in zip(axes, config.subspace_types):
    for layer_name in layer_order:
        sub = all_rows_df[(all_rows_df.layer_name == layer_name) &
                           (all_rows_df.subspace_type == subspace_type) &
                           (all_rows_df.eval_type == "frozen_recalibrated")].sort_values("k")
        ax.plot(sub.k, sub.accuracy, marker="^", label=f"{layer_name} (L{rep_layers[layer_name]})",
                color=layer_colors[layer_name])
        ax.fill_between(sub.k, sub.ci_lo, sub.ci_hi, alpha=0.15, color=layer_colors[layer_name])
        rand = all_rows_df[(all_rows_df.layer_name == layer_name) &
                            (all_rows_df.subspace_type == "random_control") &
                            (all_rows_df.eval_type == "frozen_recalibrated")].sort_values("k")
        ax.plot(rand.k, rand.accuracy, linestyle="--", color=layer_colors[layer_name], alpha=0.5)
    ax.set_title(f"{subspace_type}-class subspace")
    ax.set_xlabel("Principal components removed (k)")
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Recalibrated frozen-direction accuracy\n(same weight vector, refit threshold)")
axes[-1].legend(fontsize=8, loc="lower left")
fig.suptitle("Does the ORIGINAL DIRECTION still separate the classes, with a fair threshold?\n(compare against Fig 1 — divergence indicates a threshold-miscalibration artifact, not lost information)")
save_fig(fig, "fig1b_frozen_recalibrated_accuracy_vs_k")


In [ ]:
# Fig 2: re-fit probe accuracy vs k removed — tests whether the information
# is destroyed or just moved elsewhere in the representation.
fig, axes = plt.subplots(1, len(config.subspace_types), figsize=(6 * len(config.subspace_types), 4.5), sharey=True)
if len(config.subspace_types) == 1:
    axes = [axes]
for ax, subspace_type in zip(axes, config.subspace_types):
    for layer_name in layer_order:
        sub = all_rows_df[(all_rows_df.layer_name == layer_name) &
                           (all_rows_df.subspace_type == subspace_type) &
                           (all_rows_df.eval_type == "refit_probe")].sort_values("k")
        ax.plot(sub.k, sub.accuracy, marker="s", label=f"{layer_name} (L{rep_layers[layer_name]})",
                color=layer_colors[layer_name])
        ax.fill_between(sub.k, sub.ci_lo, sub.ci_hi, alpha=0.15, color=layer_colors[layer_name])
        rand = all_rows_df[(all_rows_df.layer_name == layer_name) &
                            (all_rows_df.subspace_type == "random_control") &
                            (all_rows_df.eval_type == "refit_probe")].sort_values("k")
        ax.plot(rand.k, rand.accuracy, linestyle="--", color=layer_colors[layer_name], alpha=0.5)
    ax.set_title(f"{subspace_type}-class subspace")
    ax.set_xlabel("Principal components removed (k)")
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Re-fit probe accuracy\n(new classifier trained on ablated activations)")
axes[-1].legend(fontsize=8, loc="lower left")
fig.suptitle("Does ANY linear separability survive after the policy subspace is removed?\n(dashed = random k-dim subspace control)")
save_fig(fig, "fig2_refit_probe_accuracy_vs_k")


In [ ]:
# Fig 3: combined trade-off — recalibrated-frozen vs. refit accuracy, within-class
# subspace, middle layer as the representative case (redundancy vs. genuine
# necessity signature). Uses frozen_recalibrated rather than raw frozen_probe,
# since the raw version's fixed threshold can saturate at chance for reasons
# unrelated to whether the direction still separates the classes (see Sec. 5).
fig, ax = plt.subplots(figsize=(7, 4.5))
for eval_type, marker, color in [("frozen_recalibrated", "^", "tab:blue"), ("refit_probe", "s", "tab:orange")]:
    sub = all_rows_df[(all_rows_df.layer_name == "middle") &
                       (all_rows_df.subspace_type == "within") &
                       (all_rows_df.eval_type == eval_type)].sort_values("k")
    ax.plot(sub.k, sub.accuracy, marker=marker, color=color,
            label="Frozen direction (recalibrated)" if eval_type == "frozen_recalibrated" else "Re-fit probe (regularized)")
    ax.fill_between(sub.k, sub.ci_lo, sub.ci_hi, alpha=0.15, color=color)
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("Principal components removed (k)")
ax.set_ylabel("Accuracy")
ax.set_title(f"Recalibrated-frozen vs. re-fit probe accuracy — middle layer (L{rep_layers['middle']}), within-class subspace\n"
             "Large gap => information is redundant/re-encoded, not destroyed")
ax.legend()
save_fig(fig, "fig3_combined_tradeoff_frozen_vs_refit")


In [ ]:
# Fig 4: layer x k heatmap of accuracy drop (baseline - ablated), RECALIBRATED
# frozen direction, within-class subspace — the raw-threshold version is
# included separately as fig4b for transparency, but this is the one that
# actually reflects whether the direction's separability was destroyed.
sub = all_rows_df[(all_rows_df.subspace_type == "within") & (all_rows_df.eval_type == "frozen_recalibrated")].copy()
sub["accuracy_drop"] = sub["baseline_accuracy"] - sub["accuracy"]
pivot = sub.pivot(index="layer_name", columns="k", values="accuracy_drop").reindex(layer_order)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="rocket_r", ax=ax, cbar_kws={"label": "Accuracy drop"})
ax.set_xlabel("Principal components removed (k)")
ax.set_ylabel("Layer")
ax.set_title("Recalibrated frozen-direction accuracy drop after within-class subspace ablation")
save_fig(fig, "fig4_layer_by_k_heatmap_frozen_within")


In [ ]:
# Fig 4b: same heatmap using the RAW (non-recalibrated) frozen-probe accuracy
# drop, kept for transparency/comparison with Fig 4 — large divergence between
# the two indicates the raw numbers are dominated by threshold miscalibration
# rather than genuine loss of separability.
sub_raw = all_rows_df[(all_rows_df.subspace_type == "within") & (all_rows_df.eval_type == "frozen_probe")].copy()
sub_raw["accuracy_drop"] = sub_raw["baseline_accuracy"] - sub_raw["accuracy"]
pivot_raw = sub_raw.pivot(index="layer_name", columns="k", values="accuracy_drop").reindex(layer_order)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
sns.heatmap(pivot_raw, annot=True, fmt=".3f", cmap="rocket_r", ax=ax, cbar_kws={"label": "Accuracy drop"})
ax.set_xlabel("Principal components removed (k)")
ax.set_ylabel("Layer")
ax.set_title("RAW (non-recalibrated) frozen-probe accuracy drop — compare against Fig 4")
save_fig(fig, "fig4b_layer_by_k_heatmap_frozen_raw_within")


In [ ]:
# Fig 5: within- vs. between-class subspace comparison at a fixed k, across layers,
# for both eval types — a compact summary panel. Uses frozen_recalibrated
# (not raw frozen_probe) for the same reason as Fig 3/Fig 4.
k_fixed = config.k_list[len(config.k_list) // 2]  # e.g. k=5 with the default k_list
sub = all_rows_df[all_rows_df.k == k_fixed]

fig, ax = plt.subplots(figsize=(8, 4.5))
width = 0.18
x = np.arange(len(layer_order))
offsets = {("within", "frozen_recalibrated"): -1.5, ("within", "refit_probe"): -0.5,
           ("between", "frozen_recalibrated"): 0.5, ("between", "refit_probe"): 1.5}
colors = {("within", "frozen_recalibrated"): "#4C72B0", ("within", "refit_probe"): "#8FA8D6",
          ("between", "frozen_recalibrated"): "#DD8452", ("between", "refit_probe"): "#F0B489"}
for (subspace_type, eval_type), off in offsets.items():
    vals = [sub[(sub.layer_name == l) & (sub.subspace_type == subspace_type) &
                (sub.eval_type == eval_type)]["accuracy"].values[0] for l in layer_order]
    label_eval = "frozen (recalibrated)" if eval_type == "frozen_recalibrated" else "refit"
    ax.bar(x + off * width, vals, width,
           label=f"{subspace_type}-class / {label_eval}",
           color=colors[(subspace_type, eval_type)])
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":")
ax.set_xticks(x)
ax.set_xticklabels([f"{l}\n(L{rep_layers[l]})" for l in layer_order])
ax.set_ylabel("Accuracy")
ax.set_title(f"Ablation comparison at k={k_fixed} components removed")
ax.legend(fontsize=8, ncol=2)
save_fig(fig, "fig5_within_vs_between_comparison_fixed_k")

print("All figures written to:", FIGURES_DIR)


## 9. Auto-generated report

Builds `REPORT.md` with the structure requested for this experiment:
Research Question, Methodology, Experimental Setup, Results, Figures,
Interpretation, Discussion, Limitations, Future Work. The report only
states what the ablation-on-probes experiment actually measured — it does
not extrapolate to generation-time behavior, which this notebook never
tests.

In [ ]:
def get_metric(layer_name, k, subspace_type, eval_type, field):
    row = all_rows_df[(all_rows_df.layer_name == layer_name) & (all_rows_df.k == k) &
                       (all_rows_df.subspace_type == subspace_type) & (all_rows_df.eval_type == eval_type)]
    return row.iloc[0][field] if len(row) else float("nan")

k_max = max(config.k_list)
report = []
report.append("# NPS Experiment 012 — Policy Subspace Causal Necessity — Report\n")
report.append("## Research Question\n")
report.append(
    "Is the policy subspace discovered in Experiment 011 causally necessary for the "
    "*linear separability* of benign vs. unsafe activations, or is the same information "
    "redundantly encoded elsewhere in the residual stream? This notebook does not test "
    "generation-time behavior (refusal, jailbreak susceptibility); it tests probe accuracy "
    "on activations with the discovered subspace projected out.\n"
)

report.append("## Methodology\n")
report.append(
    f"For each representative layer (early=L{rep_layers['early']}, middle=L{rep_layers['middle']}, "
    f"final=L{rep_layers['final']}) and each subspace definition from Experiment 011 "
    f"(within-class PCA, between-class paired-difference PCA), the top-k components "
    f"(k ∈ {config.k_list}) were projected out of held-out benign/unsafe activations "
    f"(test split reproduced with Experiment 011's exact seed/test_size). Two evaluations "
    f"were run on the ablated activations: (a) the **original, frozen** Experiment-011 "
    f"probe weight vector, and (b) a **freshly re-fit** logistic regression trained on the "
    f"ablated training split. Both were compared against ablating a random k-dimensional "
    f"subspace ({config.n_random_trials} trials, averaged) as a dimensionality-matched control.\n"
)

report.append("## Experimental Setup\n")
report.append(f"- Model: `{config.model_name}` (activations reused from Experiment 011, no generation performed)")
report.append(f"- k values: {config.k_list}")
report.append(f"- Subspace types: {config.subspace_types} + random_control")
report.append(f"- Bootstrap resamples for CIs: {config.n_bootstrap}")
report.append(f"- Random-subspace control trials: {config.n_random_trials}")
report.append(f"- Test split: seed={config.seed}, test_size={config.test_size} (matches Experiment 011)\n")

report.append("## Results\n")
report.append(
    "**Read this section's frozen-probe numbers with the methodological note below in mind.** "
    "Two tables are given for the frozen evaluation: the raw original threshold, and the "
    "recalibrated version (same weight vector, refit threshold). Where they diverge sharply, "
    "prefer the recalibrated numbers and the Interpretation section, which uses those.\n"
)
report.append("### Frozen-probe accuracy — RAW (original Exp. 011 threshold, unadjusted)\n")
report.append("| Layer | Subspace | k=" + " | k=".join(str(k) for k in config.k_list) + " |")
report.append("|---|---|" + "---|" * len(config.k_list))
for layer_name in layer_order:
    for subspace_type in config.subspace_types + ["random_control"]:
        vals = [f"{get_metric(layer_name, k, subspace_type, 'frozen_probe', 'accuracy'):.3f}" for k in config.k_list]
        report.append(f"| {layer_name} (L{rep_layers[layer_name]}) | {subspace_type} | " + " | ".join(vals) + " |")
report.append("")

report.append("### Frozen-DIRECTION accuracy — RECALIBRATED (same weight vector, refit threshold)\n")
report.append("| Layer | Subspace | k=" + " | k=".join(str(k) for k in config.k_list) + " |")
report.append("|---|---|" + "---|" * len(config.k_list))
for layer_name in layer_order:
    for subspace_type in config.subspace_types + ["random_control"]:
        vals = [f"{get_metric(layer_name, k, subspace_type, 'frozen_recalibrated', 'accuracy'):.3f}" for k in config.k_list]
        report.append(f"| {layer_name} (L{rep_layers[layer_name]}) | {subspace_type} | " + " | ".join(vals) + " |")
report.append("")

report.append("### Re-fit probe accuracy (new classifier on ablated activations, C=" + str(config.refit_probe_C) + ")\n")
report.append("| Layer | Subspace | k=" + " | k=".join(str(k) for k in config.k_list) + " |")
report.append("|---|---|" + "---|" * len(config.k_list))
for layer_name in layer_order:
    for subspace_type in config.subspace_types + ["random_control"]:
        vals = [f"{get_metric(layer_name, k, subspace_type, 'refit_probe', 'accuracy'):.3f}" for k in config.k_list]
        report.append(f"| {layer_name} (L{rep_layers[layer_name]}) | {subspace_type} | " + " | ".join(vals) + " |")
report.append("")

report.append("## Methodological Note: two evaluation artifacts and how they were handled\n")
report.append(
    "**Raw frozen-probe accuracy can saturate at exactly the test set's class balance "
    "(e.g. 0.500) even when real signal survives ablation.** If the removed component "
    "carries much of the activations' overall mean, ablation rigidly shifts every score by a "
    "constant; the original fixed intercept — calibrated for the pre-ablation distribution — "
    "can then push every test point to the same side of the decision boundary, producing a "
    "degenerate constant-class prediction. This looks identical to 'no separability survives' "
    "in the accuracy column alone, even when AUC (or a recalibrated threshold) shows real "
    "ranking signal remains. `frozen_recalibrated` fixes this by keeping the weight vector "
    "frozen but refitting a 1-D threshold on the ablated training projections.\n"
)
_n_test_example = int(get_metric(layer_order[0], config.k_list[0], config.subspace_types[0], "frozen_probe", "n_test"))
_n_train_example = int(round(_n_test_example * (1 - config.test_size) / config.test_size))
report.append(
    f"**Re-fit probe accuracy can fall below chance (not just to chance) when it is measuring "
    f"overfitting rather than information content.** With `n_train~{_n_train_example}` "
    f"and `hidden_size` in the ~1500 range, ablating even 20 dimensions leaves a severely "
    f"underdetermined (`p >> n`) regime where any training-set labeling is perfectly linearly "
    f"separable by chance. Weak regularization fits that chance structure and can generalize "
    f"worse than random guessing — a pattern that would otherwise be mistaken for strong "
    f"evidence against redundancy. `refit_probe` uses `refit_probe_C={config.refit_probe_C}` "
    f"(strong L2) specifically to keep this measurement honest; if you see accuracy well below "
    f"0.5 that continues to worsen with k even at this regularization strength, treat it as a "
    f"sign to regularize further or reduce dimensionality before fitting, not as a substantive "
    f"finding.\n"
)

# headline pattern check: does frozen drop while refit stays high (redundancy),
# or do both drop together (genuine necessity)? Uses the recalibrated frozen
# metric and the regularized refit metric — see Methodological Note above.
mid_frozen_kmax = get_metric("middle", k_max, "within", "frozen_recalibrated", "accuracy")
mid_refit_kmax  = get_metric("middle", k_max, "within", "refit_probe", "accuracy")
mid_base        = get_metric("middle", config.k_list[0], "within", "frozen_recalibrated", "baseline_accuracy")
gap = mid_refit_kmax - mid_frozen_kmax

report.append("## Interpretation\n")
report.append(
    f"At the middle representative layer, within-class subspace, removing the largest tested "
    f"subspace (k={k_max}) leaves **recalibrated** frozen-direction accuracy at "
    f"**{mid_frozen_kmax:.3f}** (baseline {mid_base:.3f}) while a freshly re-fit, strongly "
    f"regularized probe on the same ablated activations reaches **{mid_refit_kmax:.3f}** — "
    f"a gap of **{gap:+.3f}**. "
    + ("A large positive gap indicates the class-relevant information is *redundantly "
       "encoded*: the specific direction the original probe relied on was disrupted, but "
       "policy-relevant information remains linearly recoverable elsewhere in the residual "
       "stream at this layer. This means the discovered subspace is not the sole carrier of "
       "the information, even though it is the direction the trained probe happened to use."
       if gap > 0.1 else
       "A small gap indicates that once this subspace is removed, no comparably strong "
       "linear signal remains — consistent with the discovered subspace being close to "
       "causally necessary for the *linear* representation of policy information at this "
       "layer (within the scope of what this notebook tests).")
    + "\n"
)
report.append(
    "Compare each subspace-ablation row against its `random_control` row in the tables above: "
    "if random k-dimensional ablation causes a similar accuracy drop, the effect is largely a "
    "generic consequence of removing *any* k dimensions from a high-dimensional representation, "
    "not evidence that this particular subspace is special. See `statistics/statistics.json` for "
    "McNemar p-values and Cohen's h effect sizes quantifying this per condition.\n"
)

report.append("## Figures\n")
for fig_name, caption in [
    ("fig1_frozen_probe_accuracy_vs_k", "RAW frozen (original) probe accuracy vs. components removed — see Methodological Note before interpreting."),
    ("fig1b_frozen_recalibrated_accuracy_vs_k", "RECALIBRATED frozen-direction accuracy vs. components removed — the more reliable read on whether the direction still separates the classes."),
    ("fig2_refit_probe_accuracy_vs_k", "Re-fit (regularized) probe accuracy vs. components removed — tests whether information is destroyed or redundant."),
    ("fig3_combined_tradeoff_frozen_vs_refit", "Recalibrated-frozen vs. re-fit accuracy trade-off at the middle layer, within-class subspace."),
    ("fig4_layer_by_k_heatmap_frozen_within", "Layer x k heatmap of recalibrated frozen-direction accuracy drop, within-class subspace."),
    ("fig4b_layer_by_k_heatmap_frozen_raw_within", "Same heatmap using the RAW (non-recalibrated) accuracy drop, for comparison."),
    ("fig5_within_vs_between_comparison_fixed_k", f"Within- vs. between-class subspace comparison at k={k_fixed}."),
]:
    report.append(f"- `figures/{fig_name}.png` — {caption}")
report.append("")

report.append("## Discussion\n")
report.append(
    "This experiment establishes a *representational* causal claim: whether the Experiment-011 "
    "policy subspace is necessary for linear probes to separate benign/unsafe activations. It "
    "does not establish, and should not be read as establishing, anything about whether the "
    "model's generation-time behavior (refusals, compliance) causally depends on this subspace — "
    "that would require an intervention during generation, which this notebook deliberately does "
    "not perform.\n"
)

report.append("## Limitations\n")
report.append(
    "- Held-out test sets are small (governed by Experiment 011's original prompt-set size), so "
    "per-condition confidence intervals may be wide — consult `statistics.json` rather than point "
    "estimates alone.\n"
    "- Linear probes only test *linear* separability; a nonlinear probe might recover information "
    "from ablated activations that a logistic regression cannot, understating redundancy.\n"
    "- Results are specific to `Qwen/Qwen2.5-1.5B-Instruct` and the Experiment 011 prompt "
    "distribution; they may not generalize to other models or more adversarial/out-of-distribution "
    "unsafe prompts.\n"
    "- No claim is made here about whether this subspace is the *only* safety-relevant mechanism "
    "in the model — only whether it's necessary for linear separability in the tested activations.\n"
)

report.append("## Future Work\n")
report.append(
    "- If a future experiment does test generation-time behavioral effects of subspace "
    "intervention, it should be scoped and reviewed separately from this notebook, with explicit "
    "consideration of dual-use risk given that the same technique doubles as a refusal-ablation "
    "method on open-weight models.\n"
    "- Extend the redundancy check with nonlinear probes (e.g. small MLP) to see if the "
    "frozen/re-fit gap narrows or widens.\n"
    "- Repeat across model scale (7B/14B Qwen2.5) to see whether redundancy increases with model "
    "capacity.\n"
)

report_path = os.path.join(RESULTS_DIR, "REPORT.md")
with open(report_path, "w") as f:
    f.write("\n".join(report))
print("Wrote report to:", report_path)
print("\n".join(report[:20]))


## 10. Package results

Zips `results/` as `NPS_Experiment_012_Results.zip`, matching the requested
export name.

In [ ]:
import shutil

zip_base = os.path.join(BASE_DIR, "NPS_Experiment_012_Results")
zip_path = shutil.make_archive(zip_base, "zip", RESULTS_DIR)
print("Zipped results to:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))


## 11. Summary

This notebook tested whether the Experiment 011 policy subspace is
causally necessary for **linear separability** of policy-relevant
activations, by ablating it and checking (a) whether the original probe
breaks and (b) whether a freshly trained probe can still find a separating
direction — both against a random-subspace control, with bootstrap CIs and
paired significance tests.

**What this notebook intentionally did not do:** generate text, evaluate
jailbreak success, or measure refusal rate under intervention. If a
genuinely behavioral follow-up is wanted later, it should be scoped and
reviewed as its own experiment rather than folded into this one, given that
the same mechanics (activation ablation + evaluation against unsafe
prompts) is also the recipe for refusal-ablation on an open-weight model.
